# LCEL Built-in Runnables with Groq

This notebook explains the building blocks used to shape and route data inside LangChain Expression Language (LCEL).

You will learn `RunnablePassthrough`, `RunnableLambda`, `RunnableParallel`, `itemgetter`, and how these components form a small Groq RAG chain.

> Select the **Python 3.13.7 (`.venv`)** kernel and run cells from top to bottom.

## 1. Configure Groq

Production code validates required settings once, keeps secrets outside source code, and bounds timeouts, retries, and output length.

```text
GROQ_API_KEY=your_key
GROQ_MODEL=qwen/qwen3.8-27b
GROQ_MAX_TOKENS=256
GROQ_TIMEOUT_SECONDS=30
EMBEDDING_MODEL=sentence-transformers/all-MiniLM-L6-v2
```

In [1]:
import logging
import os
from dataclasses import dataclass

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(), override=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    force=True,
)
logger = logging.getLogger("builtin_runnables")


def positive_int(name: str, default: int) -> int:
    raw = os.getenv(name, str(default))
    try:
        value = int(raw)
    except ValueError as error:
        raise ValueError(f"{name} must be an integer, received {raw!r}.") from error
    if value <= 0:
        raise ValueError(f"{name} must be greater than zero.")
    return value


@dataclass(frozen=True, slots=True)
class Settings:
    groq_model: str
    embedding_model: str
    max_tokens: int
    timeout: int

    @classmethod
    def from_environment(cls) -> "Settings":
        if not os.getenv("GROQ_API_KEY"):
            raise RuntimeError("GROQ_API_KEY is missing from .env.")
        return cls(
            groq_model=os.getenv("GROQ_MODEL", "qwen/qwen3.8-27b"),
            embedding_model=os.getenv(
                "EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"
            ),
            max_tokens=positive_int("GROQ_MAX_TOKENS", 256),
            timeout=positive_int("GROQ_TIMEOUT_SECONDS", 30),
        )


settings = Settings.from_environment()

In [2]:
from langchain_groq import ChatGroq

chat_model = ChatGroq(
    model=settings.groq_model,
    temperature=0.1,
    reasoning_effort="none",
    max_tokens=settings.max_tokens,
    timeout=settings.timeout,
    max_retries=2,
)

## 2. `RunnablePassthrough`

### What?

It returns its input unchanged.

### When and why?

Use it when one branch must preserve the original value while other branches transform or retrieve data. It is common in RAG, where the original question and retrieved context are both needed.

In [3]:
from langchain_core.runnables import RunnablePassthrough

passthrough = RunnablePassthrough()
print(passthrough.invoke("Original input"))

Original input


## 3. `RunnableLambda`

### What?

It wraps a normal Python function as a Runnable.

### When and why?

Use it for deterministic validation, normalization, formatting, or data-shape conversion. Keep business rules in named, typed functions so they can be unit-tested separately.

In [4]:
from langchain_core.runnables import RunnableLambda


def normalize_name(name: str) -> str:
    clean_name = name.strip()
    if not clean_name:
        raise ValueError("Name cannot be empty.")
    if len(clean_name) > 100:
        raise ValueError("Name must not exceed 100 characters.")
    return clean_name


normalize_runnable = RunnableLambda(normalize_name)
print(normalize_runnable.invoke("  Ada Lovelace  "))

Ada Lovelace


## 4. `RunnableParallel`

### What?

It sends the same input to independent branches and returns a dictionary containing each result.

### When and why?

Use it when branches do not depend on each other. Parallel execution can reduce latency, but model branches still consume separate API requests and quota.

In [5]:
from langchain_core.runnables import RunnableParallel


def count_characters(value: str) -> int:
    return len(value)


parallel_transform = RunnableParallel(
    original=RunnablePassthrough(),
    normalized=normalize_runnable,
    character_count=RunnableLambda(count_characters),
)

print(parallel_transform.invoke("  Grace Hopper  "))

{'original': '  Grace Hopper  ', 'normalized': 'Grace Hopper', 'character_count': 16}


## 5. Shape dictionaries with `itemgetter`

`itemgetter("key")` selects one value from a dictionary. LCEL automatically treats it as a runnable inside a mapping.

Use it when a chain receives multiple fields but each downstream component needs only specific fields. A clear input contract prevents accidental key mismatches.

In [6]:
from operator import itemgetter
from typing import TypedDict

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


class FactRequest(TypedDict):
    person: str
    language: str


fact_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Give one concise, widely documented fact. Respond in {language}."),
        ("human", "Person: {person}"),
    ]
)

fact_chain = (
    {
        "person": itemgetter("person") | RunnableLambda(normalize_name),
        "language": itemgetter("language"),
    }
    | fact_prompt
    | chat_model
    | StrOutputParser()
)

print(fact_chain.invoke({"person": "Alan Turing", "language": "English"}))

2026-10-01 15:23:03,312 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Alan Turing is widely recognized as the father of theoretical computer science and artificial intelligence for his 1936 paper, which introduced the concept of the Turing machine, a mathematical model of computation that laid the foundation for modern digital computers.


## 6. Use built-in Runnables in a small RAG chain

The following example uses a small, public knowledge base so each LCEL data transformation is easy to see.

- Local Hugging Face embeddings convert text into vectors.
- Chroma performs similarity search.
- `RunnableParallel` preserves the question and retrieves context.
- Groq produces a grounded answer.

Groq does not provide embeddings, so the embedding model runs locally without paid embedding API calls.

In [8]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

knowledge_documents = [
    Document(
        page_content=(
            "Openai publishes educational content about data science, "
            "artificial intelligence, machine learning, deep learning, computer vision, "
            "natural language processing, and Python programming in English."
        ),
        metadata={"source": "demo-knowledge-base", "document_id": "profile-1"},
    )
]

embeddings = HuggingFaceEmbeddings(
    model_name=settings.embedding_model,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vector_store = Chroma.from_documents(
    documents=knowledge_documents,
    embedding=embeddings,
    ids=["profile-1"],
    collection_name="builtin_runnables_demo",
)
retriever = vector_store.as_retriever(search_kwargs={"k": 1})

2026-10-01 15:23:44,757 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-10-01 15:23:44,758 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-10-01 15:23:44,807 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
2026-10-01 15:23:45,069 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-10-01 15:23:45,123 | INFO | httpx | HTTP Request: HEAD htt

In [10]:
def format_documents(documents: list[Document]) -> str:
    if not documents:
        return "No relevant context was retrieved."
    return "\n\n".join(
        f"[Source {index}: {doc.metadata.get('source', 'unknown')}]\n{doc.page_content}"
        for index, doc in enumerate(documents, start=1)
    )


rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer only from the provided context. If the answer is absent, say you do not know.",
        ),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)

rag_chain = (
    RunnableParallel(
        question=RunnablePassthrough(),
        context=retriever | RunnableLambda(format_documents),
    )
    | rag_prompt
    | chat_model
    | StrOutputParser()
)

print(rag_chain.invoke("What topics does cover?"))

2026-10-01 15:24:14,745 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Based on the provided context, the topics covered are:

*   Data science
*   Artificial intelligence
*   Machine learning
*   Deep learning
*   Computer vision
*   Natural language processing
*   Python programming


## 7. Choosing the correct Runnable

| Need | Runnable |
| --- | --- |
| Keep the original input | `RunnablePassthrough` |
| Run a Python transformation | `RunnableLambda` |
| Run independent branches | `RunnableParallel` |
| Select a dictionary field | `itemgetter` inside an LCEL mapping |

## Enterprise guidance

- Define and validate input contracts.
- Keep deterministic transformations separate from model calls.
- Limit parallel model calls to protect rate limits and cost.
- Preserve source metadata for traceability.
- Add centralized tracing, latency, token, and error metrics.
- Use persistent, access-controlled vector storage for real data.
- Confirm data is approved before sending it to an external provider.